### Calculate the large-fire size for the FOD based on the slope where the Lorenz curve is closest to 1

In [6]:
# import libraries
import arcpy as ap
import geopandas as gpd
import numpy as np

<class 'ModuleNotFoundError'>: No module named 'geopandas'

In [1]:
## data sources
# fire occurrence database
fires = r"C:\Users\Charlie\Desktop\ArcGIS\data\NatFSim\NatFSim2022\fire_occurrence\FPA_FOD_20221014.gdb\Fires"

# pyrome shapefiles
pyrome_boundaries = r"C:\Users\Charlie\Desktop\ArcGIS\data\WCS_OPM_FY24\0_AnalysisArea\Pyromes\Pyromes_CONUS_20200206.shp"

In [2]:
# function to find minimum fire size of a pyrome
def find_min_fire_size(pyrome):

    # select individual pyrome from the shapefile and ensure it's on the same crs as the FOD
    py = pyrome_boundaries[pyrome_boundaries["PYROME"] == pyrome]
    py = py.to_crs(fires.crs) # convert to an if statement to only do the conversion if the crs are different

    # find the fires that intersect with the pyrome
    py_fires = gpd.overlay(fires, py, how='intersection')
    py_fires = py_fires[['FIRE_SIZE', 'FOD_ID']]

    # sort descending by fire size
    py_fires = py_fires.sort_values(by='FIRE_SIZE', ascending=False).reset_index(drop=True)

    # calculate cumulative fires and %
    py_fires['cum_fires'] = (np.arange(1, len(py_fires) + 1)) / len(py_fires)
    py_fires['cum_fires_per'] = py_fires['cum_fires'] * 100

    # calculate cumulative burned area and %
    py_fires['cum_area'] = py_fires['FIRE_SIZE'].cumsum() / py_fires['FIRE_SIZE'].sum()
    py_fires['cum_area_per'] = py_fires['cum_area'] * 100

    # calculate difference vectors
    # np.diff calculates the difference of each element with the element before it
    dx = np.diff(py_fires['cum_fires_per'])
    dy = np.diff(py_fires['cum_area_per'])

    # calculate slope and replace NaN with 0
    slope = dy / dx
    slope = np.insert(slope, 0, np.nan)

    # create a new dataFrame from the 2nd row onward (first calculated slope is nan)
    py_fires['slope'] = slope

    # find the row at which the slope of the line is closest to 1
    idx_1 = (py_fires['slope'] - 1).abs().idxmin() # subtract 1 from each slope value, find the absolute value, find the index position of the minimum
    row_1 = py_fires.loc[idx_1]

    return(row_1)

In [4]:
find_min_fire_size(2)

<class 'TypeError'>: string indices must be integers, not 'str'